In [2]:
import gc

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import sklearn as skt
import xgboost as xgb
import json
from scipy.stats import trim_mean
import math
from pandas.api.types import ( is_numeric_dtype, is_categorical_dtype, is_object_dtype, is_datetime64_any_dtype )
from default_risk.scripts.cv_mlfow_integration import run_cv_tracked_mlflow
from sklearn.model_selection import RepeatedKFold
import default_risk.config as cfg
import os
import xgboost as xgb
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
import dtale
import mlflow
import mlflow.xgboost
import default_risk.config
from default_risk.scripts.auxiliars_for_modeling import cast_object_into_categoricals
from default_risk.scripts.auxiliars_for_modeling import get_baseline_setup
from default_risk.scripts.auxiliars_for_modeling import prepare_columns



load_dotenv()
experiment_name = os.getenv("MLFLOW_EXPERIMENT_NAME", "default_experiment")
cv,hiperparams = get_baseline_setup()
mlflow.set_experiment(experiment_name)
mlflow.xgboost.autolog(log_models=True)



In [ ]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
prev_app_df = pd.read_parquet(cfg.PROCESSED_DIR / "previous_application.train-processed.parquet")


merged_df = application_train_df.merge(
    prev_app_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)
run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train + prev_application")
#auc_score_OOF= 0.754

#freeing memory
del merged_df
gc.collect()



[0]	validation_0-auc:0.71367
[1]	validation_0-auc:0.72475
[2]	validation_0-auc:0.72876
[3]	validation_0-auc:0.73188
[4]	validation_0-auc:0.73575
[5]	validation_0-auc:0.73836
[6]	validation_0-auc:0.74116
[7]	validation_0-auc:0.74387
[8]	validation_0-auc:0.74634
[9]	validation_0-auc:0.74739
[10]	validation_0-auc:0.74941
[11]	validation_0-auc:0.75088
[12]	validation_0-auc:0.75159
[13]	validation_0-auc:0.75233
[14]	validation_0-auc:0.75353
[15]	validation_0-auc:0.75431
[16]	validation_0-auc:0.75581
[17]	validation_0-auc:0.75612
[18]	validation_0-auc:0.75685
[19]	validation_0-auc:0.75725
[20]	validation_0-auc:0.75754
[21]	validation_0-auc:0.75786
[22]	validation_0-auc:0.75828
[23]	validation_0-auc:0.75863
[24]	validation_0-auc:0.75856
[25]	validation_0-auc:0.75874
[26]	validation_0-auc:0.75878
[27]	validation_0-auc:0.75855
[28]	validation_0-auc:0.75892
[29]	validation_0-auc:0.75900
[30]	validation_0-auc:0.75892
[31]	validation_0-auc:0.75897
[32]	validation_0-auc:0.75927
[33]	validation_0-au

442

In [4]:
#for the first experiment we gonna analize the gains from the aggregation of previous_application 
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
prev_app_installment_agg_df = pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments-2.parquet")


merged_df = application_train_df.merge(
    prev_app_installment_agg_df, 
    on="id_curr", 
    how="left"
)


#we gonna handle a lot of heavy files so we are freeing memory ASAP from now
del application_train_df, prev_app_installment_agg_df
gc.collect()

X,Y= prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train_kui + prev_application + installment")
#auc_score_OOF= 0.763

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.72806
[1]	validation_0-auc:0.73710
[2]	validation_0-auc:0.74260
[3]	validation_0-auc:0.74489
[4]	validation_0-auc:0.74757
[5]	validation_0-auc:0.74955
[6]	validation_0-auc:0.75138
[7]	validation_0-auc:0.75443
[8]	validation_0-auc:0.75680
[9]	validation_0-auc:0.75806
[10]	validation_0-auc:0.76053
[11]	validation_0-auc:0.76185
[12]	validation_0-auc:0.76212
[13]	validation_0-auc:0.76387
[14]	validation_0-auc:0.76458
[15]	validation_0-auc:0.76523
[16]	validation_0-auc:0.76601
[17]	validation_0-auc:0.76717
[18]	validation_0-auc:0.76750
[19]	validation_0-auc:0.76770
[20]	validation_0-auc:0.76772
[21]	validation_0-auc:0.76808
[22]	validation_0-auc:0.76854
[23]	validation_0-auc:0.76864
[24]	validation_0-auc:0.76843
[25]	validation_0-auc:0.76883
[26]	validation_0-auc:0.76927
[27]	validation_0-auc:0.76934
[28]	validation_0-auc:0.76928
[29]	validation_0-auc:0.76898
[30]	validation_0-auc:0.76899
[31]	validation_0-auc:0.76903
[32]	validation_0-auc:0.76991
[33]	validation_0-au

423

In [8]:
#for the second one  we gonna analize the gains from the aggregation of bureau
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_feature_engineering.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train+bureau")
#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.71040
[1]	validation_0-auc:0.72427
[2]	validation_0-auc:0.72897
[3]	validation_0-auc:0.73282
[4]	validation_0-auc:0.73461
[5]	validation_0-auc:0.73802
[6]	validation_0-auc:0.74129
[7]	validation_0-auc:0.74247
[8]	validation_0-auc:0.74465
[9]	validation_0-auc:0.74707
[10]	validation_0-auc:0.74834
[11]	validation_0-auc:0.75000
[12]	validation_0-auc:0.75156
[13]	validation_0-auc:0.75263
[14]	validation_0-auc:0.75405
[15]	validation_0-auc:0.75503
[16]	validation_0-auc:0.75559
[17]	validation_0-auc:0.75594
[18]	validation_0-auc:0.75620
[19]	validation_0-auc:0.75656
[20]	validation_0-auc:0.75669
[21]	validation_0-auc:0.75813
[22]	validation_0-auc:0.75888
[23]	validation_0-auc:0.75897
[24]	validation_0-auc:0.75938
[25]	validation_0-auc:0.75998
[26]	validation_0-auc:0.76019
[27]	validation_0-auc:0.76104
[28]	validation_0-auc:0.76108
[29]	validation_0-auc:0.76115
[30]	validation_0-auc:0.76095
[31]	validation_0-auc:0.76123
[32]	validation_0-auc:0.76121
[33]	validation_0-au

6776

In [8]:
#now bureau parent (main with feature engineering + Bureau with feature engineering + Bureau_balance)
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

del application_train_df , bureau_df
gc.collect()

X,Y = prepare_columns(merged_df)
X= cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"application_train+bureau+bureau_balance")
#auc_score_OOF= 0.759

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.72656
[1]	validation_0-auc:0.73578
[2]	validation_0-auc:0.73910
[3]	validation_0-auc:0.74179
[4]	validation_0-auc:0.74397
[5]	validation_0-auc:0.74643
[6]	validation_0-auc:0.74734
[7]	validation_0-auc:0.74874
[8]	validation_0-auc:0.75025
[9]	validation_0-auc:0.75208
[10]	validation_0-auc:0.75284
[11]	validation_0-auc:0.75531
[12]	validation_0-auc:0.75733
[13]	validation_0-auc:0.75812
[14]	validation_0-auc:0.75880
[15]	validation_0-auc:0.75977
[16]	validation_0-auc:0.76042
[17]	validation_0-auc:0.76083
[18]	validation_0-auc:0.76130
[19]	validation_0-auc:0.76179
[20]	validation_0-auc:0.76174
[21]	validation_0-auc:0.76199
[22]	validation_0-auc:0.76258
[23]	validation_0-auc:0.76299
[24]	validation_0-auc:0.76320
[25]	validation_0-auc:0.76342
[26]	validation_0-auc:0.76402
[27]	validation_0-auc:0.76414
[28]	validation_0-auc:0.76429
[29]	validation_0-auc:0.76501
[30]	validation_0-auc:0.76535
[31]	validation_0-auc:0.76513
[32]	validation_0-auc:0.76520
[33]	validation_0-au

9692

In [ ]:
#finally we try with app_train + prev_app + installments  + bureau
application_train_df = pd.read_parquet(cfg.CLEANS_DIR / "application_train.train-cleaned.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau.train-processed.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments.parquet")


merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.0 (app_train+bureau+prev_app+installments)")
#auc_score_OOF= 0.766 is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.71544
[1]	validation_0-auc:0.72608
[2]	validation_0-auc:0.73145
[3]	validation_0-auc:0.73652
[4]	validation_0-auc:0.74041
[5]	validation_0-auc:0.74480
[6]	validation_0-auc:0.74798
[7]	validation_0-auc:0.75074
[8]	validation_0-auc:0.75409
[9]	validation_0-auc:0.75600
[10]	validation_0-auc:0.75855
[11]	validation_0-auc:0.75949
[12]	validation_0-auc:0.76024
[13]	validation_0-auc:0.76184
[14]	validation_0-auc:0.76344
[15]	validation_0-auc:0.76477
[16]	validation_0-auc:0.76541
[17]	validation_0-auc:0.76641
[18]	validation_0-auc:0.76722
[19]	validation_0-auc:0.76740
[20]	validation_0-auc:0.76822
[21]	validation_0-auc:0.76844
[22]	validation_0-auc:0.76888
[23]	validation_0-auc:0.76914
[24]	validation_0-auc:0.76981
[25]	validation_0-auc:0.77018
[26]	validation_0-auc:0.77018
[27]	validation_0-auc:0.77027
[28]	validation_0-auc:0.77076
[29]	validation_0-auc:0.77120
[30]	validation_0-auc:0.77149
[31]	validation_0-auc:0.77153
[32]	validation_0-auc:0.77173
[33]	validation_0-au

In [19]:
#app_train_with_feature_engineering + prev_app + bureau + installments
application_train_df = pd.read_parquet(cfg.PROCESSED_DIR / "application_train_with_kui.parquet")
bureau_df = pd.read_parquet(cfg.PROCESSED_DIR / "bureau_parent.parquet")

merged_df = application_train_df.merge(
    bureau_df, 
    on="id_curr", 
    how="left"
)

#cleaning the first 2 df before load the third one to avoid RAM bottleneck
del application_train_df , bureau_df
gc.collect()

previous_application_df=  pd.read_parquet(cfg.PROCESSED_DIR / "prev_app_agg_installments-2.parquet")


merged_df = merged_df.merge(
    previous_application_df,
    on= "id_curr", 
    how="left"
)

#cleaning the third
del previous_application_df
gc.collect()

X,Y = prepare_columns(merged_df)
X = cast_object_into_categoricals(X)

run_cv_tracked_mlflow(xgb.XGBClassifier,hiperparams,cv,X,Y,experiment_name,"2.1 (app_train_with_features+bureau+prev_app+installments)")
#auc_score_OOF=  is the result of all the agregation at 2.0

#freeing memory
del merged_df
gc.collect()

[0]	validation_0-auc:0.72930
[1]	validation_0-auc:0.73937
[2]	validation_0-auc:0.74471
[3]	validation_0-auc:0.74682
[4]	validation_0-auc:0.74902
[5]	validation_0-auc:0.75198
[6]	validation_0-auc:0.75440
[7]	validation_0-auc:0.75715
[8]	validation_0-auc:0.75915
[9]	validation_0-auc:0.76052
[10]	validation_0-auc:0.76313
[11]	validation_0-auc:0.76497
[12]	validation_0-auc:0.76590
[13]	validation_0-auc:0.76694
[14]	validation_0-auc:0.76749
[15]	validation_0-auc:0.76874
[16]	validation_0-auc:0.76947
[17]	validation_0-auc:0.77011
[18]	validation_0-auc:0.77036
[19]	validation_0-auc:0.77168
[20]	validation_0-auc:0.77178
[21]	validation_0-auc:0.77229
[22]	validation_0-auc:0.77318
[23]	validation_0-auc:0.77361
[24]	validation_0-auc:0.77343
[25]	validation_0-auc:0.77337
[26]	validation_0-auc:0.77350
[27]	validation_0-auc:0.77372
[28]	validation_0-auc:0.77437
[29]	validation_0-auc:0.77427
[30]	validation_0-auc:0.77455
[31]	validation_0-auc:0.77466
[32]	validation_0-auc:0.77479
[33]	validation_0-au

454